<a href="https://colab.research.google.com/github/muhammad-usman-tech/data-analysis-tasks/blob/main/Python_E_commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pandas.core.reshape import encoding
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# importing
df=pd.read_csv('/content/data.csv',encoding='latin1')
# inspecting data
print('shape is :',df.shape)
# we got 5 lac 41 thousand 9 hundred and 9 rows and 8 attributes
df.info()
df.head()
# what i understood 1.(data is exploided may be on the basis of products their invoice number is telling it)
# 2.this data set have some missing values
# 3.some data types need to be changed here like date and one more(in the etl stage)

# my 5 questions if i was the owner of some store or of some e-commerce online store(after watching these attributes)
#1. which product is selling out the most(or from which stock) or which product is givving the most profit(both means different)
#2. in which country has the most sales vs the low sales
#3. i have date so i will check like is this data from which dates then i will find the trend like in which date or month we are having the most sales
#4. which month or year or day we are having the lowest sales(and why having the low sales on that specific day or date(it can be difficult question for me to find out))
#5. am i in the lost or in the profit or how must profit or loss from the day  to now



# PERFORMING THE ETL: here i this ETL i will try to complete the whole possible data cleaning techniques(that i must know for the internship)
# 1.checking the nulls
nulls=df.isnull().sum()
percentage_null=(nulls/len(df))*100
percentage_null
# ok so think about it there are 24 percent customer id are nulls and description are 0.2 percent null's
# i dont think dropping them is a good way because its only 24 percent only and replacin them with the ai generated data is also not good
# firstly customer id is the float i have to make it the int may be it will good

# df['CustomerID']=df['CustomerID'].astype('int64')
# cannot convert it into the int because there are some null here
# ok so here the best approach can be droping these rows

df=df.dropna(subset=['CustomerID','Description'])
# succesfully droped these rows
# now simple
df['CustomerID']=df['CustomerID'].astype('int64')
df.dtypes #done
nulls=df.isnull().sum()
percentage_null=(nulls/len(df))*100
percentage_null
# now nulls are completed and i want to drop and check the duplicates
print('total duplicates are where all the attributes are same :',df.duplicated().sum()) #got 5225 needs to remove these
df=df.drop_duplicates()

# confirming
print('total duplicates after droping :',df.duplicated().sum()) #got 0

# now can check some data types
df.info()
df.head()

# ok so invoice and invoice date data types need to be changed

# df['InvoiceNo']=df['InvoiceNo'].astype('int64') can't do this found some string there in that column
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'],format='mixed')
# done till here
# ok so now according to me if all the nulls are cleared and all the duplicates are cleared and the data types are almost logically correct we can just do 1 thing moere like
# doing striping and replacing any special charecter from the main attributes

# first stripping the columns name
df.columns=df.columns.str.strip()

# convert CamelCase/PascalCase to snake_case using regex and lowercasing
df.columns=df.columns.str.replace(r'(?<!^)(?=[A-Z])', '_', regex=True).str.lower()

# Display modified columns to verify
print(df.columns)

# now stripping some object columns like country and many more
obj_columns=df.select_dtypes(include='object').columns

for col in obj_columns:
    df[col]=df[col].str.strip()
#ok thats also done

for col in obj_columns:
    df[col]=df[col].str.replace(r'[^a-zA-Z0-9\s]',' ',regex=True)


# ok almost from side cleaning is done in this data set

df.to_csv('cleaned_data_e-commerce.csv')
# can import in the new cell to answer the buisness questions



shape is : (75106, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75106 entries, 0 to 75105
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    75106 non-null  object 
 1   StockCode    75106 non-null  object 
 2   Description  74903 non-null  object 
 3   Quantity     75105 non-null  float64
 4   InvoiceDate  75105 non-null  object 
 5   UnitPrice    75105 non-null  float64
 6   CustomerID   46828 non-null  float64
 7   Country      75105 non-null  object 
dtypes: float64(3), object(5)
memory usage: 4.6+ MB
total duplicates are where all the attributes are same : 715
total duplicates after droping : 0
<class 'pandas.core.frame.DataFrame'>
Index: 46113 entries, 0 to 75002
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    46113 non-null  object 
 1   StockCode    46113 non-null  object 
 2   Description  46113 non-nul

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df=pd.read_csv('/content/cleaned_data_e-commerce.csv')
# ok now start finding my answers by EDA one by one

#1. which product is selling out the most(or from which stock) or which product is givving the most profit(both means different)
# ok firstly making the column of the sale
df['total_sale']=df['quantity']*df['unit_price']

df.info()
df.head() #ok done here

# now firstly finding the which product is selling the most (can be find by their appearence or by the quantity i think quantity will be good)
# we don't have the product so we will use the stock code

selling_numers=df.groupby('stock_code')['quantity'].sum().sort_values(ascending=False)
# stock code 84077 has the most salleing quantity of 53119 with and stock code 84337 with the negative means we have more replaces of that stock insead of that selling (-1475)
# just make me sure it has more returns thats why it has negatie sign

cancelled_summary = df[df['quantity'] < 0]
cancelled_summary #yes these tells the returns

# and wat to tell we shold why these products have more returns
print('stock code with more returns :',cancelled_summary['stock_code'].nunique()) #1920 stock code with negative impact
print('numver of total unique stock code ',df['stock_code'].nunique()) #3684-1920=1764 stock codes or racks with the positive results almost half of the stock codes

# ok now thats clear now find the products with the most and least sales profit (before this understand this the unit price is the price with the profit added)

sales_point=df.groupby('stock_code')[['total_sale', 'quantity']].sum().sort_values(by='total_sale',ascending=False)
negative_sales_number=sales_point[sales_point<0]
print('numver of stock codes who gives us loss :',len(negative_sales_number)) # got 27 negative
# and the most harming stock code product was from the M	with -58745.46 loss

sales_point.head()
# stock code with the most sales is 22423	132567.70(with 11529 quantity sales) but the most selling stock was 84077 it shows that the most selling does not means it is the most profitable
# and same can be for the negative harming or returning products with the less profit

# ok so the q1 is answered the most selling product or stock code is 84077 but most profitable stock is 22423

# and randomly thought that what are the total number of the transactions done( not single single product so need to check the invoice)
df['invoice_no'].nunique() #22190 got almost 22 thosund and one hundred complete invoices

len(df) #401604 total rows are


# ok so lets do the question number 2 in this cell we will start question 3 in the new cell
# in which country has the most sales vs the low sales
# thats simple we can also make a count plot for better visualization but first we have to interprete by the prove
print('QUESTION 2====================================================================')
df['country'].nunique() #have 37 different countries thats good like we are offering services across 37 different countries
country_wise=df.groupby('country')['total_sale'].sum().sort_values(ascending=False)
country_wise.tail()
# most sales United Kingdom with sales of 6747156.154
# Saudi Arabia	sales : 131.17 it was the least sales
# notive one thing here over back regarding the stock code we were watching the negative values in sales that were representing the loss but in the country wise there is not even
# single country with the negative sales means (some stock can make loss) at over all profit is greater and that is making us in overall profit in all country wise that also tells one thing we are not in loss over all



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46113 entries, 0 to 46112
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    46113 non-null  int64  
 1   invoice_no    46113 non-null  object 
 2   stock_code    46113 non-null  object 
 3   description   46113 non-null  object 
 4   quantity      46113 non-null  float64
 5   invoice_date  46113 non-null  object 
 6   unit_price    46113 non-null  float64
 7   customer_i_d  46113 non-null  int64  
 8   country       46113 non-null  object 
 9   total_sale    46113 non-null  float64
dtypes: float64(3), int64(2), object(5)
memory usage: 3.5+ MB
stock code with more returns : 665
numver of total unique stock code  2632
numver of stock codes who gives us loss : 2632
QUESTION 2====================================================================


,total_sale
country,
Iceland,1187.18
Channel Islands,1039.11
Poland,861.38
Austria,257.04
Israel,-227.44


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df=pd.read_csv('/content/cleaned_data_e-commerce.csv')
# Q3:#3. i have date attribute so i will check like is this data from which dates then i will find the trend like
# in which date or month we are having the most sales
df['total_sales']=df['unit_price']*df['quantity']
df['invoice_date']=pd.to_datetime(df['invoice_date'],format='mixed')

print('strting date',df['invoice_date'].min()) #starting from dec 01 2010
print('ending date',df['invoice_date'].max()) #feb 11 2011
# means we have almost 2.5 months of data and its ok to be see and find the trend

# first just do the simple groupby and what we get

time_trend=df.groupby('invoice_date')['total_sales'].mean().sort_values(ascending=False)
time_trend #here we got 3146 unique values thats not good data to analyze by all the seprate dates
# we can extract the week or the days but can't by the year because our data is only 25 months 2 months and 11 days so may be by week or by day name will be ok
# if we want more detailed data so the by day name will be good according to me
# and by the way according to the time-trend the most got avg sales was the day jan 2011 and 18 date
df['day_name']=df['invoice_date'].dt.day_name()
df['day_name']
days_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['day_name']=pd.Categorical(df['day_name'],categories=days_order,ordered=True)
time_trend=df.groupby('day_name')['total_sales'].mean()
time_trend
time_trend
# .sort_values(ascending=False)
# tuesday and the friday have got the most avg it means people are interestedly buying things on tuesday and friday maybe because of some friday sales and tuesday is major working day

# but the least avg sales day are monday 19.0 and sunday 12.16 the possible reason can be people are busy on sunday or our shop timming is so much low on sunday but on monday the definately possible reason is people are much busy after the sunday on the monday
# we can do some offers on sunday and also do its advertisement for this offer on the monday

# for the more findings we can do fetch teh avg by the month but we have not complete the 3 months data it is 2 months and 11 days

# but first just create the the visualization of the day name data for ease
# and yes we got one more thing we have not any single sale on Saturday means our store is close on saturday if it is not then it really hard to think we not even a single sale on saturday
plt.figure(figsize=(8,5))
sns.lineplot(x=time_trend.index,y=time_trend.values)
plt.xlabel("DAY'S NAME")
plt.ylabel("AVG SALES")
plt.title("THE TREND OF AVG SALES ON WEEK DAYS")
plt.show()

# exact showed the same thing avg sales max on fri and tuesday and wed and thurs and almost equal and sunday and monday and the lowest
# but the graph didnt show the saturday even that was in the time_tredn data with the NAN

# again we don't have the 3 months complete so we can go for the weekly like which week have more or less sales out of the month
# df['week_name']=df['invoice_date'].dt.week
# ok so

df['week_number']=(df['invoice_date'].dt.day-1)//7+1
df['week_number'].unique()
# ok so divided into 5 weeks week 1-4 will be 7 days and week 5 will be day 29-31

week_trend=df.groupby('week_number')['total_sales'].mean()
week_trend

# interpretation or the answer of my question : week 2 has highest avg with 23. something and week three has second highest with 22 and week 1 has 3rdt position with the 21. something
# the reason is clear that these weeks are the working or salay weeks specially the week 2(8-14) which is the salary week so people prefer to buy these things in these weeks
# week 4 and 5 has avg of 17.3 and 13.7 ecause these are the month end days and people avoid purchasing in these day and wait for their salary we can advertize more things and sundays of the salary weeks with the offers and advertizement and specially
# (one imp note that the our data was 2.11 duration so we didnt that the week 3 and 4 and 5 data so this can be one reason week 1 and 2 and much higher then 4 and 5 ) but till with out the one month week 3 data we still got the week 3 avg sales at number 2 with 22.826924 greater then week 1

# now make its chart for easy understanding

plt.figure(figsize=(8,5))
sns.lineplot(x=week_trend.index,y=week_trend.values)
plt.xlabel("WEEK NUMBER")
plt.ylabel("AVG SALES")
plt.title("THE TREND OF AVG SALES ON WEEK NUMBER")
plt.show()
# SHOWED EXACTLY THE SAME THING HERE

In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


df=pd.read_csv('/content/cleaned_data_e-commerce.csv')
df.info()
df.isnull().sum()
df['invoice_date']=pd.to_datetime(df['invoice_date'],format='mixed')
df['total_sale']=df['quantity']*df['unit_price']

# #4. which month or year or day we are having the lowest sales
#  (and why having the low sales on that specific day or date(it can be difficult question for me to find out))

# ok so almost the 80 percent answer of the Q4 is founded(in Question 3) and we can just explore it little more we have found the days and the week number the more i can do is just explore with the month

df['month_name']=df['invoice_date'].dt.month_name()
df['month_name'].unique() #array(['December', 'January'], dtype=object)
# now compare the months:

df.groupby('month_name')['quantity'].sum() # dec got 2 95 thousand lac and jan got 2 lac and 52 thousand quntity sold
# now check the actual sales
df.groupby('month_name')['total_sale'].sum()
# December	552372.86
# January	446455.00
# so the jan has lower then the december
# see its difficult to find the reason without any other actual data so this can be droped because of december is a chrismis month and jan is the start of the year it is truely possible that the sale can further drop in the feb or march but the summer sales can increase it

# i tried my best to find the reasons of these ups and downs we have found in the Q3 and Q4

# now Q5: am i in the lost or in the profit or how must profit or loss from the day  to now

# first just check all that total_sales sum
df['total_sale'].sum() #998827.86 sales of almost 10 lac in 2 months and 11 days
# now i have to find the total investment and let me think

check=df[df['quantity']<=0]
check['quantity'].sum()
# np.float64(-94722.0) returned items

# but still there is no way to find the total investment or loss because returned items can have different prices but we just can have a general view
check['total_sale'].sum() #np.float64(-110918.76000000001)

check['total_sale']

# ok so after checking alot of things i got that the total sum of sales is the final profit after detucting the returned products from here and as we dont have any investment and other thing so we can not guess here


# After deep thought things:


jan_data=df[df['month_name']=='January']
jan_data.groupby('country')['total_sale'].sum().sort_values(ascending=False)



dec_data=df[df['month_name']=='December']
dec_data.groupby('country')['total_sale'].sum().sort_values(ascending=False)

# still difficult to see the only option according to me is pivot table for wide compairing


df['country'].nunique()# got 26 countries may be litttle to remember
data=pd.pivot_table(df,index='month_name',columns='country',values='total_sale',aggfunc='sum')
data

# ok so here most of the countries have less sale in jan and most sale like some countries have NAN in the jan but there are also some countries like Singapore which have nan in dec but some 1600 sale in jan (but i guess mostly countries have low sales in jan as compare to dec)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46113 entries, 0 to 46112
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    46113 non-null  int64  
 1   invoice_no    46113 non-null  object 
 2   stock_code    46113 non-null  object 
 3   description   46113 non-null  object 
 4   quantity      46113 non-null  float64
 5   invoice_date  46113 non-null  object 
 6   unit_price    46113 non-null  float64
 7   customer_i_d  46113 non-null  int64  
 8   country       46113 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 3.2+ MB


country,Australia,Austria,Belgium,Channel Islands,Cyprus,Denmark,EIRE,Finland,France,Germany,...,Lithuania,Netherlands,Norway,Poland,Portugal,Singapore,Spain,Sweden,Switzerland,United Kingdom
month_name,,,,,,,,,,,,,,,,,,,,,
December,937.60,257.04,1809.91,363.53,1587.07,1281.5,7825.57,892.80,9575.36,14527.44,...,1661.06,8784.48,3787.12,248.16,2380.12,NaN,1843.73,2646.30,1304.92,481675.23
January,8633.03,NaN,1154.05,675.58,547.50,NaN,21671.52,889.02,13965.85,14089.22,...,NaN,26476.68,NaN,613.22,3998.11,2053.07,7178.54,3446.32,3604.63,330376.59
